In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())

In [1]:
import os
import sys

sys.path.append(os.path.abspath(""))

from models.base import ModelConfig
from models.smolvlm import SmolVLM

model_path = "HuggingFaceTB/SmolVLM-Instruct"


cfg = ModelConfig.from_dict({
    "name": "smolvlm",
    "model_name": "smolvlm-instruct",
    "model_path": model_path,
    "max_new_tokens": 512,
    "temperature": 0,
    "device": "cuda",
    "device_map": "auto",
    "dtype": "bfloat16",
})

model = SmolVLM(cfg)

C:\Users\marvi\miniconda3\envs\smolvlm_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\marvi\miniconda3\envs\smolvlm_cuda\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


[LOGS] SmolVLM loaded on cuda


In [2]:
import pandas as pd

day_path = 'F:\SynDORBench\index\human_detection\day.jsonl'
day_df = pd.read_json(day_path, lines=True)

print(day_df.loc[0, 'img_path'])

images/SMPLitex_texture_00000_choking_huge_landscape_noon_south_10.8/run_pose-06-hangon-kaiwa_stageii_frame_780.npz_20250907_220206/raw_images/frame0001_cam000_r10_phi0.0_theta0.0.jpg


In [3]:
from pathlib import Path

time_instruction = f"This is 1 video frame from from a surveillance footage."

perspective_prompt = "You are seeing this image from a surveillance camera view and you are the surveillance camera. What action is the person performing?"
task_prompt = "Describe in details what you see from the video frame."

dataset_dir = Path("F:/SynDORBench")
picture_path = Path("images/SMPLitex_texture_00000_choking_huge_landscape_noon_south_10.8/run_pose-06-hangon-kaiwa_stageii_frame_780.npz_20250907_220206/raw_images/frame0001_cam000_r10_phi0.0_theta0.0.jpg")

abs_picture_path = dataset_dir / picture_path

actual_path = 'F:/SynDORBench/images/SMPLitex_texture_00000_choking_huge_landscape_noon_south_10.8/run_pose-06-hangon-kaiwa_stageii_frame_780.npz_20250907_220206/raw_images/frame0001_cam000_r10_phi0.0_theta0.0.jpg'


print(actual_path)
print(str(abs_picture_path))
print(actual_path == str(abs_picture_path))

base_text = f"\n{time_instruction}\n{perspective_prompt} {task_prompt}"

print(model.predict(actual_path, base_text))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


F:/SynDORBench/images/SMPLitex_texture_00000_choking_huge_landscape_noon_south_10.8/run_pose-06-hangon-kaiwa_stageii_frame_780.npz_20250907_220206/raw_images/frame0001_cam000_r10_phi0.0_theta0.0.jpg
F:\SynDORBench\images\SMPLitex_texture_00000_choking_huge_landscape_noon_south_10.8\run_pose-06-hangon-kaiwa_stageii_frame_780.npz_20250907_220206\raw_images\frame0001_cam000_r10_phi0.0_theta0.0.jpg
False
The person is standing on the grass. The person is wearing a red shirt and blue jeans. The person is looking up. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. The person is standing on the grass. Th

In [ ]:
day_df.head()

In [ ]:
def get_inference_from_df(model, df, root_dir="", limit=5):
    results = []

    # for prototyping, limit to head
    if limit:
        df = df.head(limit)

    for i in range(len(df)):
        img_path = df.loc[i, 'img_path']
        task_prompt = df.loc[i, 'prompt']
        if root_dir:
            img_path = root_dir / img_path

        pred = model.predict(img_path, task_prompt)


        results.append({
            "img_path": img_path,
            "task_prompt": task_prompt,
            "prediction": pred
        })
    return pd.DataFrame(results)


small_df = day_df.head(100)
result_df = get_inference_from_df(model, small_df, root_dir=dataset_dir)

In [ ]:
result_df